In [ ]:
#@title shopify products parser (parallel + retry + failure log)

import requests
import json
import csv
import time
from typing import List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed


# ---------------- GLOBAL CONFIG ----------------
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

MAX_RETRIES = 3
BACKOFF_FACTOR = 2
TIMEOUT = 10
MAX_WORKERS = 5  # adjust based on speed vs rate-limit


# ---------------- GET ALL PRODUCT HANDLES ----------------
def get_products_url(website):
    base_url = f"https://{website}/products.json"
    all_products_data = []
    page = 1
    limit = 100

    while True:
        params = {"page": page, "limit": limit}

        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()

        products = data.get("products", [])

        if not products:
            break

        for product in products:
            all_products_data.append({
                "id": product["id"],
                "handle": product["handle"]
            })

        if len(products) < limit:
            break

        page += 1

    print(f"[INFO] Found {len(all_products_data)} products.")
    return all_products_data


# ---------------- FETCH FUNCTION (MULTITHREAD WORKER) ----------------
def fetch_product(args):
    """
    args = (handle, url)
    returns: (handle, url, json_data, error)
    """
    handle, url = args

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
            return (handle, url, response.json(), None)

        except Exception as e:
            print(f"[RETRY {attempt}/{MAX_RETRIES}] {handle} failed: {e}")

            if attempt == MAX_RETRIES:
                return (handle, url, None, str(e))

            sleep_time = BACKOFF_FACTOR ** (attempt - 1)
            time.sleep(sleep_time)

    return (handle, url, None, "Unknown error")


# ---------------- PARSE FUNCTION ----------------
def parse_product(product: Dict) -> List[Dict]:
    results = []

    product_data = {
        "product_id": product.get("id"),
        "title": product.get("title"),
        "handle": product.get("handle"),
        "description": product.get("description"),
        "created_at": product.get("created_at"),
        "vendor": product.get("vendor"),
        "type": product.get("type"),
        "price": product.get("price"),
        "compare_at_price": product.get("compare_at_price"),
        "available": product.get("available"),
    }

    variants = product.get("variants", [])

    if not variants:
        results.append(product_data)
        return results

    for variant in variants:
        row = product_data.copy()

        row.update({
            "variant_id": variant.get("id"),
            "variant_title": variant.get("title"),
            "variant_sku": variant.get("sku"),
            "variant_available": variant.get("available"),
            "variant_name": variant.get("name"),
            "variant_price": variant.get("price"),
            "variant_compare_at_price": variant.get("compare_at_price"),
            "variant_barcode": variant.get("barcode"),
            "variant_image": variant.get("featured_image", {}).get("src") if variant.get("featured_image") else None
        })

        results.append(row)

    return results


# ---------------- MAIN ----------------
def main(website):

    all_products = get_products_url(website)

    BASE_URL = f"https://{website}/products/{{}}.js"
    OUTPUT_JSON = f"{website}_products.json"
    OUTPUT_CSV = f"{website}_products.csv"
    FAILED_LOG = f"{website}_failed_urls.json"

    all_data = []
    failed_urls = []

    # prepare tasks
    tasks = []
    for item in all_products:
        handle = item["handle"]
        url = BASE_URL.format(handle)
        tasks.append((handle, url))

    print(f"[INFO] Starting parallel fetch with {MAX_WORKERS} workers...")

    # ---------------- MULTITHREAD EXECUTION ----------------
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(fetch_product, task) for task in tasks]

        for future in as_completed(futures):
            handle, url, product_json, error = future.result()

            if error or not product_json:
                failed_urls.append({
                    "handle": handle,
                    "url": url,
                    "error": error
                })
                continue

            parsed_rows = parse_product(product_json)
            all_data.extend(parsed_rows)

    # ---------------- SAVE OUTPUT ----------------
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_data, f, ensure_ascii=False, indent=2)

    if all_data:
        keys = all_data[0].keys()
        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(all_data)

    with open(FAILED_LOG, "w", encoding="utf-8") as f:
        json.dump(failed_urls, f, indent=2)

    print(f"[DONE] Saved {len(all_data)} rows")
    print(f"[FAILED] {len(failed_urls)} URLs logged to {FAILED_LOG}")

In [ ]:
# ---------------- RUN ----------------
if __name__ == "__main__":
    main("moustaphalabban.com/")

Streaming output truncated to the last 5000 lines.
[RETRY 1/3] jimmy-choo-women-edp failed: 403 Client Error: Forbidden for url: https://moustaphalabban.com//products/jimmy-choo-women-edp.js
[RETRY 1/3] cartier-ladies-la-panthere-parfum failed: 403 Client Error: Forbidden for url: https://moustaphalabban.com//products/cartier-ladies-la-panthere-parfum.js
[RETRY 1/3] givenchy-irresistible-rose-velvet-women-edp failed: 403 Client Error: Forbidden for url: https://moustaphalabban.com//products/givenchy-irresistible-rose-velvet-women-edp.js
[RETRY 2/3] cartier-pasha-edition-noir-absolu-parfum failed: 403 Client Error: Forbidden for url: https://moustaphalabban.com//products/cartier-pasha-edition-noir-absolu-parfum.js
[RETRY 2/3] cartier-pasha-noir-absolu-parfum-limited-edition failed: 403 Client Error: Forbidden for url: https://moustaphalabban.com//products/cartier-pasha-noir-absolu-parfum-limited-edition.js
[RETRY 2/3] jimmy-choo-women-edp failed: 403 Client Error: Forbidden for url: htt